[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-2-ml-dl-essentials/02-how-models-learn/code/loss_function_gradient_descent.ipynb)

# Class 2.2: How models learn

The slides carry the intuition. Here you make gradient descent concrete: compute a loss, take steps by hand, plot the descent, see how the learning rate changes everything, and compare two losses.

What we will cover:
- mean squared error on a small example
- why classification uses cross-entropy, not MSE
- gradient descent on a curve, plotted step by step
- the learning rate: converge, crawl, or diverge
- fitting a line to data with the same loop


## Setup

You need matplotlib for the plots in this notebook (numpy is already installed from Module 1). To install it in your virtual environment, run:

```
pip install matplotlib
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Mean squared error, in code

MSE averages the squared gap between prediction and truth. One big miss dominates, because squaring grows fast.

In [ ]:
y_hat = np.array([2.0, 4.0, 6.0, 8.0])
y     = np.array([2.5, 3.5, 6.5, 7.0])

errors = y_hat - y
mse = np.mean(errors**2)
print("errors        :", errors)
print("squared errors:", errors**2)
print("MSE           :", round(float(mse), 3))

## Why classification uses cross-entropy

A classifier outputs a probability. Suppose the true label is 1. MSE barely punishes a confident-wrong guess; cross-entropy punishes it hard. Watch the two columns diverge as the prediction gets worse.

In [ ]:
def mse_one(p, y=1.0):
    return (y - p)**2

def bce_one(p, y=1.0):
    p = min(max(p, 1e-9), 1 - 1e-9)          # keep the log finite
    return -(y*np.log(p) + (1-y)*np.log(1-p))

print(f"{'p':>6} {'MSE':>8} {'cross-entropy':>15}")
for p in [0.9, 0.5, 0.1, 0.01]:
    print(f"{p:>6} {mse_one(p):>8.3f} {bce_one(p):>15.3f}")

## Gradient descent on a curve

Minimize `f(x) = x**2`, whose slope is `2*x`, from `x = 5` with a learning rate of 0.3. Each step moves opposite the slope, and the moves shrink near the bottom.

In [ ]:
def f(x):  return x**2
def grad(x): return 2*x

x = 5.0
lr = 0.3
path = [x]
for step in range(12):
    x = x - lr*grad(x)
    path.append(x)

print("x each step:", [round(v, 3) for v in path])
print("final x    :", round(path[-1], 4), " (true minimum is 0)")

grid = np.linspace(-5.5, 5.5, 200)
plt.figure(figsize=(6, 4))
plt.plot(grid, f(grid), color="#2563eb", label="f(x) = x^2")
plt.plot(path, [f(v) for v in path], "o-", color="#b45309", label="descent path")
plt.xlabel("x (the knob)"); plt.ylabel("loss"); plt.legend(); plt.title("Rolling to the bottom")
plt.show()

## The learning rate decides everything

Same curve, three learning rates. Too small crawls, just right converges, too large diverges (the loss blows up).

In [ ]:
def run(lr, x0=5.0, steps=20):
    x, losses = x0, []
    for _ in range(steps):
        losses.append(f(x))
        x = x - lr*grad(x)
    return losses

plt.figure(figsize=(6, 4))
for lr, name in [(0.02, "0.02 crawls"), (0.4, "0.4 converges"), (1.05, "1.05 diverges")]:
    plt.plot(run(lr), "o-", label=name)
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log scale)")
plt.legend(); plt.title("Converge, crawl, or diverge")
plt.show()

for lr in [0.02, 0.4, 1.05]:
    losses = run(lr)
    if losses[-1] > losses[0]:
        verdict = "diverges"
    elif losses[-1] < 0.1:
        verdict = "converges"
    else:
        verdict = "crawls (still far from 0)"
    print(f"lr={lr:<5} start loss={losses[0]:>8.3f}  final loss={losses[-1]:>12.3f}  -> {verdict}")

## Fit a line with the same loop

Now the knobs are a slope `w` and an intercept `b`. We descend on the MSE between `w*x + b` and the data, and recover the line that made it.

In [ ]:
rng = np.random.default_rng(0)
xs = np.arange(10, dtype=float)
ys = 2.0*xs + 1.0 + rng.normal(0, 0.5, size=xs.shape)   # true line: slope 2, intercept 1

w, b, lr = 0.0, 0.0, 0.01
n = len(xs)
for _ in range(6000):
    pred = w*xs + b
    err = pred - ys
    w -= lr * (2/n) * np.sum(err*xs)
    b -= lr * (2/n) * np.sum(err)

print("learned slope w :", round(w, 3), " (true 2.0)")
print("learned intercept b:", round(b, 3), " (true 1.0)")

plt.figure(figsize=(6, 4))
plt.scatter(xs, ys, color="#2563eb", label="data")
plt.plot(xs, w*xs + b, color="#b45309", label=f"fit: {w:.2f}x + {b:.2f}")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("A line, learned by descent")
plt.show()

## Your turn

**Micro-assignment.** Six problems on loss and descent; see `../micro-assignment/README.md`.

**Next, class 2.3 (Neural networks):** one neuron, then layers, then softmax and cross-entropy for classification.